# Sistemas continuos y discretos

:::{important .simple icon=false} **Sistema**
Proceso por el cual las señales de entrada son transformadas, o provocan una respuesta, dando lugar a otras señales como salidas.
:::

Según si las señales de entrada y salida son continuas o discretas, se habla de:

- **Sistemas continuos**: Señales continuas de entrada son transformadas en señales continuas de salida: $x(t)\ \rightarrow y(t)$.

```{figure} figures/T1/2_2_fig1  
---
width: 60%
---
```

- **Sistemas discretos**: Señales discretas de entrada son transformadas en señales discretas de salida: $x[n]\ \rightarrow y[n]$.

```{figure} figures/T1/2_2_fig2 
---
width: 60%
---
```



Además, existen sistemas que tienen entrada continua y salida discreta (muestreadores) y viceversa (interpoladores), que veremos en el tema de muestreo (tema 7).

## Ejemplos sencillos de sistemas

Una de las ventajas del estudio de sistemas es que sistemas muy distintos físicamente producen expresiones matemáticas similares.

### Ejemplos de sistemas continuos:

#### Circuito RC

```{figure} figures/T1/2_2_fig3 
---
width: 60%
---
```


Considerando $v_s(t)$ como la señal de entrada y $v_c(t)$ como la señal de salida, obtenemos:
```{math}
v_s(t)=R i(t)+v_c(t) \quad \Rightarrow\quad i(t)=\frac{v_s(t)-v_c(t)}{R},
```
```{math}
i(t)=C\frac{d v_c(t)}{dt}.
```
Por tanto, se obtiene la siguiente ecuación diferencial que relaciona la salida con la entrada del sistema:
```{math}
\frac{d v_c(t)}{dt}+\frac{1}{RC}v_c(t)=\frac{1}{RC}v_s(t).
```


In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(".."))

from utils.plot_helpers import style_math_axes, add_math_ticks

In [2]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Slider, Select, Div
from bokeh.layouts import column, row

output_notebook(verbose=False, hide_banner=True);

# ==============================================================================
# 1. DATOS PRECALCULADOS
# ==============================================================================
t_min, t_max = 0, 10
t = np.linspace(t_min, t_max, 2000)
dt = t[1] - t[0]

R_values = np.round(np.linspace(0.2, 5.0, 49), 2)
C_values = np.round(np.linspace(0.2, 5.0, 49), 2)

input_types = ["Escalón", "Seno", "Pulso", "Cuadrada"]

def make_input(t, input_type):
    if input_type == "Escalón":
        return np.ones_like(t)
    elif input_type == "Seno":
        return np.sin(2*np.pi*0.5*t)
    elif input_type == "Pulso":
        return np.where((t >= 1) & (t <= 3), 1.0, 0.0)
    elif input_type == "Cuadrada":
        return np.sign(np.sin(2*np.pi*0.5*t))

def simulate_rc(vs, R, C):
    tau = R*C
    vc = np.zeros_like(vs)

    for n in range(1, len(vs)):
        vc[n] = vc[n-1] + dt*(vs[n-1] - vc[n-1])/tau

    return vc

# Precalculamos solo para R y C iniciales
R_init = 1.0
C_init = 1.0
input_init = "Escalón"

vs_init = make_input(t, input_init)
vc_init = simulate_rc(vs_init, R_init, C_init)

source = ColumnDataSource(data=dict(
    t=t,
    vs=vs_init,
    vc=vc_init
))

# ==============================================================================
# 2. FIGURA
# ==============================================================================

p = figure(
    height=400,
    width=600,
    tools="pan,wheel_zoom,reset",
    title="Circuito RC: respuesta temporal"
)

p.line("t", "vs", source=source, line_width=3, color="black",
       alpha=0.45, legend_label="Entrada $v_s(t)$")

p.line("t", "vc", source=source, line_width=4, color="blue",
       legend_label="Salida $v_c(t)$")

p.grid.grid_line_alpha = 0.25
p.legend.location = "center_right"
p.legend.click_policy = "hide"

style_math_axes(
    p,
    x_range=(t_min, t_max),
    y_range=(0, 1),
    xlabel="t",
    ylabel="voltaje"
)

add_math_ticks(
    p,
    yticks=[0, 1],
    ytick_labels=["0", "1"],
    tick_len=5
)

# ==============================================================================
# 3. CONTROLES
# ==============================================================================

R_slider = Slider(start=0.2, end=5.0, value=R_init, step=0.1, title="Resistencia R")
C_slider = Slider(start=0.2, end=5.0, value=C_init, step=0.1, title="Capacidad C")

input_select = Select(
    title="Entrada",
    value=input_init,
    options=input_types
)

info = Div(width=600)

info.text = f"""
<div style="font-family:sans-serif; font-size:14px;">
<b>Circuito RC</b><br>
Ecuación del sistema:
<br>
<span style="font-size:18px;">
dv<sub>c</sub>(t)/dt + 1/(RC) v<sub>c</sub>(t) = 1/(RC) v<sub>s</sub>(t)
</span>
<br><br>
Constante de tiempo:
<span style="font-size:18px;">τ = RC = {R_init*C_init:.2f}</span>
</div>
"""

# ==============================================================================
# 4. CALLBACK JAVASCRIPT
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        R_slider=R_slider,
        C_slider=C_slider,
        input_select=input_select,
        info=info
    ),
    code="""
    const data = source.data;
    const t = data["t"];
    const vs = data["vs"];
    const vc = data["vc"];

    const R = R_slider.value;
    const C = C_slider.value;
    const tau = R*C;
    const input_type = input_select.value;

    const dt = t[1] - t[0];

    // Entrada v_s(t)
    for (let i = 0; i < t.length; i++) {
        if (input_type === "Escalón") {
            vs[i] = 1.0;
        }
        else if (input_type === "Seno") {
            vs[i] = Math.sin(2*Math.PI*0.5*t[i]);
        }
        else if (input_type === "Pulso") {
            vs[i] = (t[i] >= 1.0 && t[i] <= 3.0) ? 1.0 : 0.0;
        }
        else if (input_type === "Cuadrada") {
            vs[i] = Math.sign(Math.sin(2*Math.PI*0.5*t[i]));
        }
    }

    // Salida v_c(t)
    vc[0] = 0.0;

    for (let i = 1; i < t.length; i++) {
        vc[i] = vc[i-1] + dt*(vs[i-1] - vc[i-1])/tau;
    }

    info.text = `
    <div style="font-family:sans-serif; font-size:14px;">
    <b>Circuito RC</b><br>
    Ecuación del sistema:
    <br>
    <span style="font-size:18px;">
    dv<sub>c</sub>(t)/dt + 1/(RC) v<sub>c</sub>(t) = 1/(RC) v<sub>s</sub>(t)
    </span>
    <br><br>
    Constante de tiempo:
    <span style="font-size:18px;">τ = RC = ${tau.toFixed(2)}</span>
    </div>
    `;

    source.change.emit();
    """
)

R_slider.js_on_change("value", callback)
C_slider.js_on_change("value", callback)
input_select.js_on_change("value", callback)

# ==============================================================================
# 5. LAYOUT
# ==============================================================================

controls = column(input_select, R_slider, C_slider, width=280)

layout = column(
    p, controls,
    info
)

show(layout)

#### Objeto móvil

```{figure} figures/T1/2_2_fig4 
```


Si consideramos $f(t)$ como la señal de entrada y $v(t)$ como la señal de salida, $m$ la masa del objeto y $\rho v$ la resistencia por fricción:
```{math}
f(t)=m a(t)+\rho v(t), \qquad a(t)=\frac{d v(t)}{dt}.
```
```{math}
f(t)=m\frac{d v(t)}{dt}+\rho v(t).
```
Se obtiene la siguiente ecuación diferencial para el sistema:
```{math}
\frac{d v(t)}{dt}+\frac{\rho}{m}v(t)=\frac{1}{m}f(t).
```

Ambos ejemplos son matemáticamente equivalentes, pues tienen la misma ecuación diferencial, que podemos escribir:
```{math}
\frac{d y(t)}{dt}+a y(t)=b x(t),
```
siento $x(t)$ la señal de entrada al sistema, $y(t)$ la señal de salida y $a$ y $b$ constantes.


In [3]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Slider, Select, Div
from bokeh.layouts import column, row

output_notebook(verbose=False, hide_banner=True);

# ==============================================================================
# 1. DATOS INICIALES
# ==============================================================================
t = np.linspace(0, 10, 2000)
dt = t[1] - t[0]

m_init = 1.0
rho_init = 1.0
input_init = "Escalón"

def make_force(t, input_type):
    if input_type == "Escalón":
        return np.ones_like(t)
    elif input_type == "Seno":
        return np.sin(2*np.pi*0.5*t)
    elif input_type == "Pulso":
        return np.where((t >= 1) & (t <= 3), 1.0, 0.0)
    elif input_type == "Cuadrada":
        return np.sign(np.sin(2*np.pi*0.5*t))

def simulate_mass(f, m, rho):
    v = np.zeros_like(f)
    for n in range(1, len(f)):
        v[n] = v[n-1] + dt*(f[n-1] - rho*v[n-1])/m
    return v

f_init = make_force(t, input_init)
v_init = simulate_mass(f_init, m_init, rho_init)

source = ColumnDataSource(data=dict(
    t=t,
    f=f_init,
    v=v_init
))

# ==============================================================================
# 2. FIGURA
# ==============================================================================

p = figure(
    height=400,
    width=600,
    tools="pan,wheel_zoom,reset",
    title="Objeto móvil con fricción: respuesta temporal"
)

p.line("t", "f", source=source, line_width=3, color="black",
       alpha=0.45, legend_label="Entrada f(t)")

p.line("t", "v", source=source, line_width=4, color="blue",
       legend_label="Salida v(t)")

p.grid.grid_line_alpha = 0.25
p.legend.location = "center_right"
p.legend.click_policy = "hide"

style_math_axes(
    p,
    x_range=(t_min, t_max),
    y_range=(0, 1),
    xlabel="t",
    ylabel="amplitud"
)

add_math_ticks(
    p,
    yticks=[0, 1],
    ytick_labels=["0", "1"],
    tick_len=5
)

# ==============================================================================
# 3. CONTROLES
# ==============================================================================

m_slider = Slider(start=0.2, end=5.0, value=m_init, step=0.1, title="Masa m")
rho_slider = Slider(start=0.2, end=5.0, value=rho_init, step=0.1, title="Fricción ρ")

input_select = Select(
    title="Entrada",
    value=input_init,
    options=["Escalón", "Seno", "Pulso", "Cuadrada"]
)

info = Div(width=600)

info.text = f"""
<div style="font-family:sans-serif; font-size:14px;">
<b>Objeto móvil con fricción</b><br>
Ecuación del sistema:
<br>
<span style="font-size:18px;">
dv(t)/dt + ρ/m v(t) = 1/m f(t)
</span>
<br><br>
Constante característica:
<span style="font-size:18px;">τ = m/ρ = {m_init/rho_init:.2f}</span>
</div>
"""

# ==============================================================================
# 4. CALLBACK JAVASCRIPT
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        m_slider=m_slider,
        rho_slider=rho_slider,
        input_select=input_select,
        info=info
    ),
    code="""
    const data = source.data;
    const t = data["t"];
    const f = data["f"];
    const v = data["v"];

    const m = m_slider.value;
    const rho = rho_slider.value;
    const tau = m/rho;
    const input_type = input_select.value;

    const dt = t[1] - t[0];

    // Entrada f(t)
    for (let i = 0; i < t.length; i++) {
        if (input_type === "Escalón") {
            f[i] = 1.0;
        }
        else if (input_type === "Seno") {
            f[i] = Math.sin(2*Math.PI*0.5*t[i]);
        }
        else if (input_type === "Pulso") {
            f[i] = (t[i] >= 1.0 && t[i] <= 3.0) ? 1.0 : 0.0;
        }
        else if (input_type === "Cuadrada") {
            f[i] = Math.sign(Math.sin(2*Math.PI*0.5*t[i]));
        }
    }

    // Salida v(t)
    // m dv/dt + rho v = f
    // dv/dt = (f - rho v)/m

    v[0] = 0.0;

    for (let i = 1; i < t.length; i++) {
        v[i] = v[i-1] + dt*(f[i-1] - rho*v[i-1])/m;
    }

    info.text = `
    <div style="font-family:sans-serif; font-size:14px;">
    <b>Objeto móvil con fricción</b><br>
    Ecuación del sistema:
    <br>
    <span style="font-size:18px;">
    dv(t)/dt + ρ/m v(t) = 1/m f(t)
    </span>
    <br><br>
    Constante característica:
    <span style="font-size:18px;">τ = m/ρ = ${tau.toFixed(2)}</span>
    <br>
    Para una entrada escalón f(t)=1, la velocidad final tiende a:
    <span style="font-size:18px;">v∞ = 1/ρ = ${(1/rho).toFixed(2)}</span>
    </div>
    `;

    source.change.emit();
    """
)

m_slider.js_on_change("value", callback)
rho_slider.js_on_change("value", callback)
input_select.js_on_change("value", callback)

# ==============================================================================
# 5. LAYOUT
# ==============================================================================

controls = column(input_select, m_slider, rho_slider, width=280)

layout = column(
    p, controls,
    info
)

show(layout)

### Ejemplos de sistemas discretos

#### Cuenta de ahorro en un banco al final de cada año:

Se obtiene la siguiente ecuación en diferencias:
```{math}
y[n]=1.01 y[n-1]+x[n],
```
o de forma equivalente,
```{math}
y[n]-1.01y[n-1]=x[n].
```

#### Simulación digital del objeto móvil:

Tomamos el tiempo en intervalos de longitud $\Delta$: 
```{math}
t=n\Delta.
```
Aproximamos la derivada mediante la primera diferencia:
```{math}
\frac{dv(t)}{dt}\simeq\frac{v(n\Delta)-v((n-1)\Delta)}{\Delta}.
```
Por tanto, la ecuación diferencial queda:
```{math}
\frac{v(n\Delta)-v((n-1)\Delta)}{\Delta}+\frac{\rho}{m}v(n\Delta)=\frac{1}{m}f(n\Delta),
```
usando las siguientes definiciones de señales discretas a partir de las continuas muestreadas:
```{math}
\begin{cases}v[n]=v(n\Delta),\\f[n]=f(n\Delta),\end{cases}
```
obtenemos:
```{math}
v[n]-v[n-1]+\frac{\rho\Delta}{m}v[n]=\frac{\Delta}{m}f[n],
```
y agrupando términos:
```{math}
v[n]\frac{m+\rho\Delta}{m}-v[n-1]=\frac{\Delta}{m}f[n].
```
Si finalmente dividimos la ecuación por el factor $(m+\rho\Delta)/m$:
```{math}
v[n]-\frac{m}{m+\rho\Delta}v[n-1]=\frac{\Delta}{m+\rho\Delta}f[n].
```

:::{tip .simple}
Se puede observar que ambos son ejemplos del mismo tipo de sistema, ya que tienen la misma ecuación en diferencias:
```{math}
y[n]-a y[n-1]=b x[n].
```


## Sistemas elementales
Un concepto fundamental en el análisis de señales y sistemas es el de la transformación de una señal mediante un cierto sistema. Vamos a ver algunas transformaciones elementales de señales, que serán muy importantes en el resto de la asignatura, y nos permitirán entender mejor algunas propiedades de las señales, ya vistas, como son las señales pares e impares, reales e imaginarias, hermíticas y antihermíticas (ver la seccción [](#clases_senales)), señales periódicas (ver la sección [](#periodicas)), y de sistemas, que veremos en la sección 2. [](#propiedades_sistemas).

En primer lugar veremos algunas transformaciones elementales sobre la amplitud de la señal, o sumas y diferencias sobre la señal:

### Cambio de nivel 
:::{important .simple icon=false} Cambio de nivel 
Multiplicación de la señal por una constante.
:::

```{figure} figures/T1/2_2_fig5_a 
---
width: 60%
---
```

```{figure} figures/T1/2_2_fig5_b 
---
width: 60%
---
```


Amplificación: $A>1$.
Atenuación: $A<1$.

### Conjugación

```{figure} figures/T1/2_2_fig6_a 
---
width: 60%
---
```

```{figure} figures/T1/2_2_fig6_b 
---
width: 60%
---
```


Se ha usado en señales reales e imaginarias y hermíticas y antihermíticas.

### Integración

```{figure} figures/T1/2_2_fig7_a 
---
width: 60%
---
```


### Sumación o acumulación
:::{important .simple icon=false} Sumación o acumulación
Equivalente a la integración para señales discretas.
:::

```{figure} figures/T1/2_2_fig7_b 
---
width: 60%
---
```



### Derivación
Operación inversa a la integración.

```{figure} figures/T1/2_2_fig8_a 
---
width: 60%
---
```



### Diferenciación o primera diferencia
Operación inversa a la sumación.

```{figure} figures/T1/2_2_fig8_b 
---
width: 70%
---
```


---

In [4]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Select, Div, Slider
from bokeh.layouts import column, row

from utils.plot_helpers import style_math_axes

output_notebook(verbose=False, hide_banner=True)

# ==============================================================================
# 1. EJE TEMPORAL
# ==============================================================================

t_min, t_max = -5.0, 5.0
num_points = 2000
t = np.linspace(t_min, t_max, num_points)

initial_system = "Amplificador"
A0 = 1.0
param0 = 2.0

# ==============================================================================
# 2. SEÑAL DE ENTRADA
# ==============================================================================

x0 = np.exp(-0.5*t**2) * np.cos(2*np.pi*0.5*t)

def compute_output(name, x, t, param):
    if name == "Amplificador":
        y = param*x

    elif name == "Atenuador":
        y = x/param

    elif name == "Sumador constante":
        y = x + param

    elif name == "Rectificador":
        y = np.abs(x)

    elif name == "Limitador":
        y = np.clip(x, -param, param)

    elif name == "Derivador":
        y = np.gradient(x, t)

    elif name == "Integrador":
        dt = t[1] - t[0]
        y = np.cumsum(x)*dt

    return y

y0 = compute_output(initial_system, x0, t, param0)

source = ColumnDataSource(data=dict(
    t=t,
    x=x0,
    y=y0,
    zero=np.zeros_like(t),
))

# ==============================================================================
# 3. FIGURA
# ==============================================================================

p = figure(
    height=400,
    width=750,
    title="Sistemas elementales: entrada x(t) y salida y(t)",
    tools="pan,wheel_zoom,reset"
)

p.line("t", "x", source=source, line_width=3, color="navy",
       legend_label="entrada x(t)")

p.line("t", "y", source=source, line_width=3, color="firebrick",
       legend_label="salida y(t)")

p.line("t", "zero", source=source, color="black", alpha=0.4)

p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.grid.grid_line_alpha = 0.25

style_math_axes(
    p,
    x_range=(t_min, t_max),
    y_range=(-3.5, 3.5),
    xlabel="t",
    ylabel="amplitud"
)

# ==============================================================================
# 4. CONTROLES
# ==============================================================================

select = Select(
    title="Sistema",
    value=initial_system,
    options=[
        "Amplificador",
        "Atenuador",
        "Sumador constante",
        "Rectificador",
        "Limitador",
        "Derivador",
        "Integrador",
    ],
    width=240
)

amp_slider = Slider(
    title="Amplitud de entrada A",
    start=0.2,
    end=2.0,
    value=A0,
    step=0.1,
    width=220
)

param_slider = Slider(
    title="Parámetro del sistema",
    start=0.2,
    end=3.0,
    value=param0,
    step=0.1,
    width=220
)

info = Div(width=750)

texts = {
    "Amplificador": "y(t) = K x(t)",
    "Atenuador": "y(t) = x(t)/K",
    "Sumador constante": "y(t) = x(t) + K",
    "Rectificador": "y(t) = |x(t)|",
    "Limitador": "y(t) = clip(x(t), -K, K)",
    "Derivador": "y(t) = dx(t)/dt",
    "Integrador": "y(t) = ∫ x(τ)dτ",
}

info.text = f"""
<b>{initial_system}</b><br>
<span style="font-size:18px;">{texts[initial_system]}</span><br><br>
La curva azul es la entrada y la curva roja es la salida del sistema.
"""

# ==============================================================================
# 5. CALLBACK
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        select=select,
        amp_slider=amp_slider,
        param_slider=param_slider,
        info=info,
        texts=texts
    ),
    code="""
    const data = source.data;
    const t = data["t"];

    const name = select.value;
    const A = amp_slider.value;
    const K = param_slider.value;

    const x = [];
    const y = [];

    for (let i = 0; i < t.length; i++) {
        const ti = t[i];
        const xi = A*Math.exp(-0.5*ti*ti)*Math.cos(2*Math.PI*0.5*ti);
        x.push(xi);
    }

    for (let i = 0; i < t.length; i++) {
        let yi = 0;

        if (name === "Amplificador") {
            yi = K*x[i];
        }

        else if (name === "Atenuador") {
            yi = x[i]/K;
        }

        else if (name === "Sumador constante") {
            yi = x[i] + K;
        }

        else if (name === "Rectificador") {
            yi = Math.abs(x[i]);
        }

        else if (name === "Limitador") {
            yi = Math.max(-K, Math.min(K, x[i]));
        }

        else if (name === "Derivador") {
            if (i === 0) {
                yi = (x[i+1] - x[i])/(t[i+1] - t[i]);
            }
            else if (i === t.length - 1) {
                yi = (x[i] - x[i-1])/(t[i] - t[i-1]);
            }
            else {
                yi = (x[i+1] - x[i-1])/(t[i+1] - t[i-1]);
            }
        }

        else if (name === "Integrador") {
            let integral = 0;
            for (let j = 0; j < i; j++) {
                const dt = t[j+1] - t[j];
                integral += 0.5*(x[j] + x[j+1])*dt;
            }
            yi = integral;
        }

        y.push(yi);
    }

    data["x"] = x;
    data["y"] = y;

    info.text = `
    <b>${name}</b><br>
    <span style="font-size:18px;">${texts[name]}</span><br><br>
    La curva azul es la entrada y la curva roja es la salida del sistema.
    `;

    source.change.emit();
    """
)

select.js_on_change("value", callback)
amp_slider.js_on_change("value", callback)
param_slider.js_on_change("value", callback)

layout = column(
    p,
    row(select, amp_slider, param_slider),
    info
)

show(layout)


## Transformación de la variable independiente

A continuación veremos operaciones elementales realizadas sobre la variable independiente.

### Desplazamiento o corrimiento en el tiempo
  
- Continuo:
  
```{figure} figures/T1/2_2_fig9_a 
---
width: 70%
---
```


- Discreto:
  
```{figure} figures/T1/2_2_fig9_b 
```

Se ha usado para definir las señales periódicas.


### Inversión en el tiempo o abatimiento

:::{important .simple icon=false} Inversión en el tiempo o abatimiento
Reflexión respecto al origen de tiempos.
:::

```{figure} figures/T1/2_2_fig10 
```

Se ha usado para definir señales pares e impares y hermíticas y antihermíticas.

**Ejemplo**: cinta escuchada al revés.
  
### Cambio de escala
  
- Continuo:
  
```{figure} figures/T1/2_2_fig11_a 
```
  
Si $x(t_0)=0,\quad at=t_0\ \Rightarrow\ t=t_0/a$.
  
- Discreto:
  
```{figure} figures/T1/2_2_fig11_b 
---
width: 75%
---
```


La expansión discreta o inserción de ceros se define de la siguiente forma:
\begin{equation}
    x[n/k]\overset{\Delta}{=}\begin{cases}
x[n/k], & n=Mk,\\
0, & \text{resto.}
    \end{cases},\qquad M\in\Z.
\end{equation}

**Ejemplo**: cambio de velocidad en disco de vinilo.

---


In [5]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Select, Div, Slider
from bokeh.layouts import column, row

from utils.plot_helpers import style_math_axes

output_notebook(verbose=False, hide_banner=True)

# ==============================================================================
# 1. EJE TEMPORAL
# ==============================================================================

t_min, t_max = -6.0, 6.0
num_points = 2500
t = np.linspace(t_min, t_max, num_points)

initial_transformation = "Desplazamiento temporal"
A0 = 1.0
a0 = 1.5

# ==============================================================================
# 2. SEÑAL BASE COMPLEJA NO PAR
# ==============================================================================

def x_real_base(u):
    return np.exp(-0.35*(u + 1.0)**2) * np.cos(2*np.pi*0.45*u)

def x_imag_base(u):
    return 0.6*np.exp(-0.25*(u - 0.8)**2) * np.sin(2*np.pi*0.65*u)

x_real0 = A0*x_real_base(t)
x_imag0 = A0*x_imag_base(t)

y_real0 = A0*x_real_base(t - a0)
y_imag0 = A0*x_imag_base(t - a0)

source = ColumnDataSource(data=dict(
    t=t,
    x_real=x_real0,
    x_imag=x_imag0,
    y_real=y_real0,
    y_imag=y_imag0,
    zero=np.zeros_like(t),
))

# ==============================================================================
# 3. FIGURA
# ==============================================================================

p = figure(
    height=400,
    width=600,
    title="Transformaciones de variable",
    tools="pan,wheel_zoom,reset"
)

p.line("t", "x_real", source=source, line_width=3, color="navy",
       legend_label="Re{x(t)}")

p.line("t", "x_imag", source=source, line_width=2, color="navy",
       line_dash="dashed", alpha=0.65,
       legend_label="Im{x(t)}")

p.line("t", "y_real", source=source, line_width=3, color="darkorange",
       legend_label="Re{y(t)}")

p.line("t", "y_imag", source=source, line_width=2, color="darkorange",
       line_dash="dashed", alpha=0.65,
       legend_label="Im{y(t)}")

p.line("t", "zero", source=source, color="black", alpha=0.4)

p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.grid.grid_line_alpha = 0.25

style_math_axes(
    p,
    x_range=(t_min, t_max),
    y_range=(-2.2, 2.2),
    xlabel="t",
    ylabel="amplitud"
)

# ==============================================================================
# 4. CONTROLES
# ==============================================================================

select = Select(
    title="Transformación",
    value=initial_transformation,
    options=[
        "Desplazamiento temporal",
        "Adelanto temporal",
        "Inversión temporal",
        "Escalado temporal",
        "Compresión temporal",
        "Expansión temporal",
        "Escalado de amplitud",
        "Conjugación compleja",
    ],
    width=190
)

amp_slider = Slider(
    title="Amplitud A",
    start=0.2,
    end=2.0,
    value=A0,
    step=0.1,
    width=190
)

param_slider = Slider(
    title="Parámetro a",
    start=0.2,
    end=4.0,
    value=a0,
    step=0.1,
    width=190
)

info = Div(width=600)

texts = {
    "Desplazamiento temporal": "y(t) = x(t - a)",
    "Adelanto temporal": "y(t) = x(t + a)",
    "Inversión temporal": "y(t) = x(-t)",
    "Escalado temporal": "y(t) = x(a t)",
    "Compresión temporal": "y(t) = x(a t), con a > 1",
    "Expansión temporal": "y(t) = x(t/a), con a > 1",
    "Escalado de amplitud": "y(t) = a x(t)",
    "Conjugación compleja": "y(t) = x*(t)",
}

info.text = f"""
<b>{initial_transformation}</b><br>
<span style="font-size:18px;">{texts[initial_transformation]}</span><br><br>
Línea continua: parte real. Línea discontinua: parte imaginaria.
"""

# ==============================================================================
# 5. CALLBACK
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        select=select,
        amp_slider=amp_slider,
        param_slider=param_slider,
        info=info,
        texts=texts
    ),
    code="""
    const data = source.data;
    const t = data["t"];

    const name = select.value;
    const A = amp_slider.value;
    const a = param_slider.value;

    const x_real = [];
    const x_imag = [];
    const y_real = [];
    const y_imag = [];

    function xreal(u) {
        return Math.exp(-0.35*(u + 1.0)*(u + 1.0))
             * Math.cos(2*Math.PI*0.45*u);
    }

    function ximag(u) {
        return 0.6*Math.exp(-0.25*(u - 0.8)*(u - 0.8))
             * Math.sin(2*Math.PI*0.65*u);
    }

    for (let i = 0; i < t.length; i++) {

        const ti = t[i];

        const xr = A*xreal(ti);
        const xi = A*ximag(ti);

        let yr = xr;
        let yi = xi;

        if (name === "Desplazamiento temporal") {
            yr = A*xreal(ti - a);
            yi = A*ximag(ti - a);
        }

        else if (name === "Adelanto temporal") {
            yr = A*xreal(ti + a);
            yi = A*ximag(ti + a);
        }

        else if (name === "Inversión temporal") {
            yr = A*xreal(-ti);
            yi = A*ximag(-ti);
        }

        else if (name === "Escalado temporal") {
            yr = A*xreal(a*ti);
            yi = A*ximag(a*ti);
        }

        else if (name === "Compresión temporal") {
            yr = A*xreal(a*ti);
            yi = A*ximag(a*ti);
        }

        else if (name === "Expansión temporal") {
            yr = A*xreal(ti/a);
            yi = A*ximag(ti/a);
        }

        else if (name === "Escalado de amplitud") {
            yr = a*A*xreal(ti);
            yi = a*A*ximag(ti);
        }

        else if (name === "Conjugación compleja") {
            yr = xr;
            yi = -xi;
        }

        x_real.push(xr);
        x_imag.push(xi);
        y_real.push(yr);
        y_imag.push(yi);
    }

    data["x_real"] = x_real;
    data["x_imag"] = x_imag;
    data["y_real"] = y_real;
    data["y_imag"] = y_imag;

    info.text = `
    <b>${name}</b><br>
    <span style="font-size:18px;">${texts[name]}</span><br><br>
    Línea continua: parte real. Línea discontinua: parte imaginaria.
    `;

    source.change.emit();
    """
)

select.js_on_change("value", callback)
amp_slider.js_on_change("value", callback)
param_slider.js_on_change("value", callback)

layout = column(
    p,
    row(select, amp_slider, param_slider),
    info
)

show(layout)

In [6]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Slider, Select, Div
from bokeh.layouts import column, row

from utils.plot_helpers import style_math_axes, add_math_ticks

output_notebook(verbose=False, hide_banner=True)

# ==============================================================================
# 1. EJE DISCRETO
# ==============================================================================

n_min, n_max = -14, 14
n = np.arange(n_min, n_max + 1)

alpha0 = "1"
beta0 = 0

# ==============================================================================
# 2. SEÑAL BASE DISCRETA
# ==============================================================================

def x_base(k):
    return 1.2*np.exp(-0.18*(k + 3)**2)

x0 = x_base(n)
y0 = x_base(n)

source = ColumnDataSource(data=dict(
    n=n,
    x=x0,
    y=y0,
    zero=np.zeros_like(n),
))

# ==============================================================================
# 3. FIGURA
# ==============================================================================

p = figure(
    height=450,
    width=600,
    title="Transformación discreta: y[n] = x[αn + β]",
    tools="pan,wheel_zoom,reset"
)

p.segment(
    "n", "zero", "n", "x",
    source=source,
    line_width=2,
    color="navy",
    alpha=0.45
)

p.scatter(
    "n", "x",
    source=source,
    size=9,
    color="navy",
    legend_label="x[n]"
)

p.segment(
    "n", "zero", "n", "y",
    source=source,
    line_width=3,
    color="firebrick"
)

p.scatter(
    "n", "y",
    source=source,
    size=10,
    color="firebrick",
    legend_label="y[n]"
)

p.line(
    "n", "zero",
    source=source,
    color="black",
    alpha=0.35
)

p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.grid.grid_line_alpha = 0.25

style_math_axes(
    p,
    x_range=(n_min - 1, n_max + 1),
    y_range=(-2.2, 2.2),
    xlabel="n",
    ylabel="amplitud"
)

add_math_ticks(
    p,
    yticks=[-1, 1],
    ytick_labels=["-1", "1"],
    tick_len=5
)

# ==============================================================================
# 4. CONTROLES
# ==============================================================================

alpha_select = Select(
    title="α",
    value=alpha0,
    options=[
        "-3",
        "-2",
        "-1",
        "-0.5",
        "-0.25",
        "0.25",
        "0.5",
        "1",
        "2",
        "3",
    ],
    width=190
)

beta_slider = Slider(
    title="β",
    start=-10,
    end=10,
    value=beta0,
    step=1,
    width=190
)

info = Div(width=600)

info.text = """
<b>Transformación discreta</b><br>
<span style="font-size:18px;">y[n] = x[αn + β]</span><br><br>

En tiempo discreto, la compresión puede ser destructiva porque
muchas muestras originales dejan de utilizarse.<br><br>

Prueba, por ejemplo:<br>
α = 2 → compresión destructiva<br>
α = 0.5 → expansión con huecos
"""

# ==============================================================================
# 5. CALLBACK
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        alpha_select=alpha_select,
        beta_slider=beta_slider,
        info=info
    ),
    code="""
    const data = source.data;
    const n = data["n"];

    const alpha = parseFloat(alpha_select.value);
    const beta = beta_slider.value;

    const x = [];
    const y = [];

    function xbase(k) {
        return (
            1.2*Math.exp(-0.18*(k + 3)*(k + 3))
        );
    }

    const used_indices = [];

    for (let i = 0; i < n.length; i++) {
        const ni = n[i];

        x.push(xbase(ni));

        const u = alpha*ni + beta;

        if (Math.abs(alpha) < 1 && alpha !== 0) {

            if (Number.isInteger(u)) {
                y.push(xbase(u));
                used_indices.push(u);
            }

            else {
                y.push(0);
            }
        }

        else {
            y.push(xbase(u));
            used_indices.push(u);
        }
    }

    data["x"] = x;
    data["y"] = y;

    const unique_indices = [...new Set(used_indices)];

    let explanation = "";

    if (alpha === 0) {
        explanation = `
        <b style="color:firebrick;">α = 0</b><br>
        Toda la señal se colapsa en una constante:
        <span style="font-size:17px;">y[n] = x[β]</span>.
        `;
    }

    else if (Math.abs(alpha) > 1) {
        explanation = `
        <b style="color:firebrick;">Compresión discreta destructiva</b><br>
        Al usar <span style="font-size:17px;">y[n] = x[${alpha}n ${beta >= 0 ? "+" : "-"} ${Math.abs(beta)}]</span>,
        solo se toman algunas muestras de la señal original.<br><br>

        Por ejemplo, si α = 2, se usan índices pares:
        <span style="font-size:17px;">..., -4, -2, 0, 2, 4, ...</span><br><br>

        Las muestras intermedias se pierden.
        `;
    }

    else if (Math.abs(alpha) < 1) {
        explanation = `
        <b style="color:seagreen;">Expansión discreta</b><br>
        La señal se alarga, pero aparecen huecos porque
        no todos los valores de <span style="font-size:17px;">αn + β</span>
        son enteros.<br><br>

        Esos huecos aparecen como muestras ausentes en la señal roja.
        Para rellenarlos habría que interpolar o definir una regla adicional.
        `;
    }

    else if (alpha < 0) {
        explanation = `
        <b>Inversión temporal</b><br>
        Como α &lt; 0, la señal se invierte en el tiempo.
        `;
    }

    else {
        explanation = `
        <b>Desplazamiento discreto</b><br>
        Con α = 1, la transformación es:
        <span style="font-size:17px;">y[n] = x[n + β]</span>.
        `;
    }

    let beta_text = "";

    if (beta !== 0) {
        beta_text = "<br><br>Además, β ≠ 0 produce un desplazamiento discreto.";
    }

    info.text = `
    <b>Transformación discreta</b><br>
    <span style="font-size:18px;">
    y[n] = x[${alpha}n ${beta >= 0 ? "+" : "-"} ${Math.abs(beta)}]
    </span><br><br>

    ${explanation}
    ${beta_text}
    `;

    source.change.emit();
    """
)

alpha_select.js_on_change("value", callback)
beta_slider.js_on_change("value", callback)

# ==============================================================================
# 6. LAYOUT
# ==============================================================================

layout = column(
    p,
    row(alpha_select, beta_slider),
    info
)

show(layout)

### Transformación lineal del eje de tiempos

```{figure} figures/T1/2_2_fig12 
---
width: 60%
---
```

Conserva la forma de la señal, pero:

  - $|\alpha|<1$: alarga linealmente la señal.
  - $|\alpha|>1$: comprime linealmente la señal.
  - $\alpha<0$: invierte en el tiempo la señal.
  - $\beta\neq 0$: desplaza en el tiempo la señal.

:::{warning} Atención
Forma de hacerlo gráficamente de forma sistemática[^1]:
 [^1]:Se puede hacer en el orden inverso pero hay que tener cuidado.

  1. Desplazamiento: $x(t)\ \rightarrow\ x(t+\beta)$.
  2. Escalamiento y/o inversión: $x(t+\beta)\ \rightarrow\ x(\alpha t+\beta)$.
:::

**Ejemplo**: $x(t)\ \rightarrow\ x\left(\frac{3}{2}t+1\right).$

```{figure} figures/T1/2_2_fig13 
---
width: 60%
---
```


Podemos comprobar que en ciertos instantes temporales de interés el resultado es correcto:

-  $t=-\frac{2}{3}\ \Rightarrow\ x\left(\frac{3}{2}\cdot\left(-\frac{2}{3}\right)+1\right)=x(-1+1)=x(0);$
-  $t=0\ \Rightarrow\ x\left(\frac{3}{2}\cdot 0+1\right)=x(1);$
-  $ t=\frac{2}{3}\ \Rightarrow\ x\left(\frac{3}{2}\cdot\frac{2}{3}+1\right)=x(1+1)=x(2).$

---

In [7]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Slider, Div
from bokeh.layouts import column, row

from utils.plot_helpers import style_math_axes

output_notebook(verbose=False, hide_banner=True)

# ==============================================================================
# 1. EJES
# ==============================================================================

t_min, t_max = -5.0, 5.0
num_points = 3000
t = np.linspace(t_min, t_max, num_points)

alpha0 = 1.5
beta0 = 1.0

# ==============================================================================
# 2. SEÑAL BASE NO PAR
# ==============================================================================

def x_base(u):
    return (
        1.1*np.exp(-0.7*(u + 1.0)**2)
        - 0.7*np.exp(-1.2*(u - 1.4)**2)
        + 0.25*np.exp(-0.4*u**2)*np.sin(2*np.pi*0.8*u)
    )

x0 = x_base(t)
y0 = x_base(alpha0*t + beta0)
z0 = x_base(t + beta0)

# Puntos de referencia del ejemplo: u = alpha*t + beta
u_ref = np.array([0.0, 1.0, 2.0])
t_ref = (u_ref - beta0)/alpha0
y_ref = x_base(u_ref)

source = ColumnDataSource(data=dict(
    t=t,
    x=x0,
    z=z0,
    y=y0,
    zero=np.zeros_like(t),
))

points = ColumnDataSource(data=dict(
    t_ref=t_ref,
    u_ref=u_ref,
    y_ref=y_ref,
    labels=[
        "t = -2/3 → x(0)",
        "t = 0 → x(1)",
        "t = 2/3 → x(2)",
    ]
))

# ==============================================================================
# 3. FIGURA
# ==============================================================================

p = figure(
    height=420,
    width=600,
    title="Transformación lineal del eje de tiempos: y(t) = x(αt + β)",
    tools="pan,wheel_zoom,reset"
)

p.line("t", "x", source=source, line_width=3, color="navy",
       legend_label="x(t) original")

p.line("t", "z", source=source, line_width=2, color="gray",
       line_dash="dashed", alpha=0.8,
       legend_label="paso 1: x(t + β)")

p.line("t", "y", source=source, line_width=3, color="firebrick",
       legend_label="y(t) = x(αt + β)")

p.scatter("t_ref", "y_ref", source=points, size=10, color="darkorange",
          legend_label="puntos de referencia")

p.segment("t_ref", 0, "t_ref", "y_ref", source=points,
          color="darkorange", line_dash="dotted", alpha=0.8)

p.line("t", "zero", source=source, color="black", alpha=0.4)

p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.grid.grid_line_alpha = 0.25

style_math_axes(
    p,
    x_range=(t_min, t_max),
    y_range=(-1.4, 1.6),
    xlabel="t",
    ylabel="amplitud"
)

# ==============================================================================
# 4. CONTROLES
# ==============================================================================

alpha_slider = Slider(
    title="α: escalamiento / inversión",
    start=-3.0,
    end=3.0,
    value=alpha0,
    step=0.1,
    width=280
)

beta_slider = Slider(
    title="β: desplazamiento",
    start=-3.0,
    end=3.0,
    value=beta0,
    step=0.1,
    width=280
)

info = Div(width=600)

info.text = """
<b>Transformación:</b>
<span style="font-size:18px;">y(t) = x(αt + β)</span><br><br>

Para el ejemplo inicial:
<span style="font-size:18px;">y(t) = x(1.5t + 1)</span><br><br>

Los puntos naranjas muestran que:<br>
t = -2/3 → y(t) = x(0)<br>
t = 0 → y(t) = x(1)<br>
t = 2/3 → y(t) = x(2)
"""

# ==============================================================================
# 5. CALLBACK
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        points=points,
        alpha_slider=alpha_slider,
        beta_slider=beta_slider,
        info=info
    ),
    code="""
    const data = source.data;
    const t = data["t"];

    const alpha = alpha_slider.value;
    const beta = beta_slider.value;

    const x = [];
    const z = [];
    const y = [];

    function xbase(u) {
        return (
            1.1*Math.exp(-0.7*(u + 1.0)*(u + 1.0))
            - 0.7*Math.exp(-1.2*(u - 1.4)*(u - 1.4))
            + 0.25*Math.exp(-0.4*u*u)*Math.sin(2*Math.PI*0.8*u)
        );
    }

    for (let i = 0; i < t.length; i++) {
        const ti = t[i];

        x.push(xbase(ti));
        z.push(xbase(ti + beta));
        y.push(xbase(alpha*ti + beta));
    }

    data["x"] = x;
    data["z"] = z;
    data["y"] = y;

    const u_ref = [0, 1, 2];
    const t_ref = [];
    const y_ref = [];
    const labels = [];

    for (let i = 0; i < u_ref.length; i++) {
        const u = u_ref[i];

        if (Math.abs(alpha) > 1e-8) {
            const tref = (u - beta)/alpha;
            t_ref.push(tref);
            y_ref.push(xbase(u));
            labels.push(`t = ${(tref).toFixed(2)} → x(${u})`);
        }
    }

    points.data["t_ref"] = t_ref;
    points.data["u_ref"] = u_ref;
    points.data["y_ref"] = y_ref;
    points.data["labels"] = labels;

    let effect = "";

    if (Math.abs(alpha) < 1) {
        effect += "|α| < 1: la señal se alarga. ";
    } else if (Math.abs(alpha) > 1) {
        effect += "|α| > 1: la señal se comprime. ";
    } else {
        effect += "|α| = 1: no hay cambio de escala. ";
    }

    if (alpha < 0) {
        effect += "Además, α < 0 produce inversión temporal. ";
    }

    if (beta !== 0) {
        effect += "β ≠ 0 produce desplazamiento temporal.";
    }

    info.text = `
    <b>Transformación:</b>
    <span style="font-size:18px;">y(t) = x(αt + β)</span><br><br>

    <b>Valores actuales:</b><br>
    α = ${alpha.toFixed(2)}<br>
    β = ${beta.toFixed(2)}<br><br>

    <b>Interpretación:</b><br>
    ${effect}<br><br>

    <b>Regla de correspondencia:</b><br>
    Para saber qué valor de x aparece en un instante t de la señal transformada:<br>
    <span style="font-size:18px;">u = αt + β</span><br>
    es decir, <span style="font-size:18px;">y(t) = x(u)</span>.
    `;

    source.change.emit();
    points.change.emit();
    """
)

alpha_slider.js_on_change("value", callback)
beta_slider.js_on_change("value", callback)

layout = column(
    p,
    row(alpha_slider, beta_slider),
    info
)

show(layout)

--- 

:::{warning .simple icon=false} Ejercicio 1
Ajusta los parámetros y el orden de transformación para obtener la función objetivo
:::

In [8]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Slider, Select, Div
from bokeh.layouts import column, row

from utils.plot_helpers import style_math_axes, add_math_ticks

output_notebook(verbose=False, hide_banner=True)

# ==============================================================================
# 1. EJES Y PARÁMETROS DEL EJERCICIO
# ==============================================================================

t_min, t_max = -5.0, 5.0
num_points = 3000
t = np.linspace(t_min, t_max, num_points)

# Señal objetivo: y(t) = x(alpha*t + beta)
alpha_target = 1.5
beta_target = 1.0

alpha0 = 1.0
beta0 = 0.0
order0 = "1) Desplazar y luego escalar"

# ==============================================================================
# 2. SEÑAL BASE NO PAR
# ==============================================================================

def x_base(u):
    return (
        1.1*np.exp(-0.7*(u + 1.0)**2)
        - 0.7*np.exp(-1.2*(u - 1.4)**2)
        + 0.25*np.exp(-0.4*u**2)*np.sin(2*np.pi*0.8*u)
    )

x0 = x_base(t)
target0 = x_base(alpha_target*t + beta_target)

student0 = x_base(alpha0*t + beta0)
intermediate0 = x_base(t + beta0)

source = ColumnDataSource(data=dict(
    t=t,
    x=x0,
    target=target0,
    student=student0,
    intermediate=intermediate0,
    zero=np.zeros_like(t),
))

# ==============================================================================
# 3. FIGURA
# ==============================================================================

p = figure(
    height=430,
    width=600,
    title="Ejercicio: obtener y(t) = x(1.5t + 1)",
    tools="pan,wheel_zoom,reset"
)

p.line("t", "x", source=source, line_width=3, color="navy",
       legend_label="x(t) original")

p.line("t", "target", source=source, line_width=4, color="black",
       alpha=0.45, legend_label="objetivo: x(1.5t + 1)")

p.line("t", "intermediate", source=source, line_width=2, color="gray",
       line_dash="dashed", alpha=0.8, legend_label="paso intermedio")

p.line("t", "student", source=source, line_width=3, color="firebrick",
       legend_label="tu resultado")

p.line("t", "zero", source=source, color="black", alpha=0.35)

p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.grid.grid_line_alpha = 0.25

style_math_axes(
    p,
    x_range=(t_min, t_max),
    y_range=(-1.5, 1.5),
    xlabel="t",
    ylabel="amplitud"
)

add_math_ticks(
    p,
    yticks=[-1, 1],
    ytick_labels=["-1", "1"],
    tick_len=5
)

# ==============================================================================
# 4. CONTROLES
# ==============================================================================

alpha_slider = Slider(
    title="α",
    start=-3.0,
    end=3.0,
    value=alpha0,
    step=0.1,
    width=190
)

beta_slider = Slider(
    title="β",
    start=-3.0,
    end=3.0,
    value=beta0,
    step=0.1,
    width=190
)

order_select = Select(
    title="Orden de transformación",
    value=order0,
    options=[
        "1) Desplazar y luego escalar",
        "2) Escalar y luego desplazar",
    ],
    width=220
)

info = Div(width=600)

info.text = """
<b>Ejercicio</b><br>
Ajusta los parámetros y el orden de transformación para obtener:<br>
<span style="font-size:18px;">y(t) = x(1.5t + 1)</span><br><br>

Recuerda el procedimiento recomendado:<br>
1. Primero desplazar: <span style="font-size:17px;">x(t) → x(t + β)</span><br>
2. Después escalar: <span style="font-size:17px;">x(t + β) → x(αt + β)</span>
"""

# ==============================================================================
# 5. CALLBACK
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        alpha_slider=alpha_slider,
        beta_slider=beta_slider,
        order_select=order_select,
        info=info,
        alpha_target=alpha_target,
        beta_target=beta_target
    ),
    code="""
    const data = source.data;
    const t = data["t"];

    const alpha = alpha_slider.value;
    const beta = beta_slider.value;
    const order = order_select.value;

    const x = [];
    const target = [];
    const student = [];
    const intermediate = [];

    function xbase(u) {
        return (
            1.1*Math.exp(-0.7*(u + 1.0)*(u + 1.0))
            - 0.7*Math.exp(-1.2*(u - 1.4)*(u - 1.4))
            + 0.25*Math.exp(-0.4*u*u)*Math.sin(2*Math.PI*0.8*u)
        );
    }

    for (let i = 0; i < t.length; i++) {
        const ti = t[i];

        x.push(xbase(ti));
        target.push(xbase(alpha_target*ti + beta_target));

        let inter;
        let out;

        if (order === "1) Desplazar y luego escalar") {
            inter = xbase(ti + beta);
            out = xbase(alpha*ti + beta);
        }

        else {
            inter = xbase(alpha*ti);
            out = xbase(alpha*(ti + beta));
        }

        intermediate.push(inter);
        student.push(out);
    }

    data["x"] = x;
    data["target"] = target;
    data["intermediate"] = intermediate;
    data["student"] = student;

    let error = 0;
    for (let i = 0; i < t.length; i++) {
        const diff = student[i] - target[i];
        error += diff*diff;
    }
    error = Math.sqrt(error/t.length);

    let message = "";

    if (error < 0.02) {
        message = "<b style='color:green;'>Correcto.</b> Has obtenido prácticamente la señal objetivo.";
    } else {
        message = "<b style='color:firebrick;'>Todavía no coincide.</b> Ajusta α, β y el orden.";
    }

    info.text = `
    <b>Ejercicio</b><br>
    Objetivo:
    <span style="font-size:18px;">y(t) = x(1.5t + 1)</span><br><br>

    Tus valores:<br>
    α = ${alpha.toFixed(2)}<br>
    β = ${beta.toFixed(2)}<br>
    Orden: ${order}<br><br>

    Error respecto al objetivo: ${error.toFixed(4)}<br>
    ${message}<br><br>

    <b>Pista:</b><br>
    Fíjate en que incluso usando los valores correctos de α y β,
    si cambias el orden de las transformaciones,
    el resultado ya no coincide con la señal objetivo.<br><br>

    En general:
    <span style="font-size:17px;">x(αt + β) ≠ x(α(t + β))</span>
    `;

    source.change.emit();
    """
)

alpha_slider.js_on_change("value", callback)
beta_slider.js_on_change("value", callback)
order_select.js_on_change("value", callback)

# ==============================================================================
# 6. LAYOUT
# ==============================================================================

layout = column(
    p,
    row(alpha_slider, beta_slider, order_select),
    info
)

show(layout)